# S13 — RNNs and LSTMs

**Week 7 · Wed Oct 7, 2026 · Module 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s13_rnns_and_lstms.ipynb)

Every cell below is a worked example from the [S13 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s13/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s13.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s13.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## Truncated BPTT, demonstrated


*Expected output starts with:* `dependency length LAG = 4, chance accuracy = 0.250`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Truncated BPTT on a task with a known dependency length. The input is a
# stream of one-hot symbols; the target at step t is the symbol seen at
# step t - LAG, so the model must hold each symbol for LAG steps. We train
# with several truncation windows, always carrying the hidden state
# (detached) across window boundaries. A gradient path from a prediction
# back to the symbol it depends on exists only when both fall inside the
# same window -- which requires window > LAG.

V, LAG, HIDDEN = 4, 4, 64

def train(window, n_windows=1200):
    torch.manual_seed(0)
    cell = nn.LSTMCell(V, HIDDEN)
    head = nn.Linear(HIDDEN, V)
    opt = torch.optim.Adam(list(cell.parameters()) + list(head.parameters()), lr=1e-2)
    h = torch.zeros(1, HIDDEN); c = torch.zeros(1, HIDDEN)
    history = torch.zeros(LAG, dtype=torch.long)  # the last LAG symbols seen
    correct = total = 0
    for w in range(n_windows):
        h, c = h.detach(), c.detach()   # carry the state, not the gradient
        loss = 0.0
        for _ in range(window):
            sym = torch.randint(0, V, (1,))
            x = torch.zeros(1, V); x[0, sym] = 1.0
            h, c = cell(x, (h, c))
            logits = head(h)
            target = history[0].view(1)          # the symbol from LAG steps ago
            loss = loss + nn.functional.cross_entropy(logits, target)
            if w >= n_windows - 100:             # accuracy over the last 100 windows
                correct += int(logits.argmax(dim=-1).item() == target.item())
                total += 1
            history = torch.cat([history[1:], sym])
        opt.zero_grad(); loss.backward(); opt.step()
    return correct / total

print(f"dependency length LAG = {LAG}, chance accuracy = {1/V:.3f}")
for window in [2, 4, 8, 16]:
    acc = train(window)
    tag = "window > LAG" if window > LAG else "window <= LAG"
    print(f"  truncation window {window:>2} ({tag:>13}): accuracy = {acc:.3f}")

## Vanishing and exploding gradients, with real numbers


*Expected output starts with:* `Linear recurrence: dh_T/dh_1 = w^(T-1)`


In [ ]:
import numpy as np

np.random.seed(0)

# Part 1: linear recurrence h[t] = w * h[t-1] + x[t].
# The gradient dh_T/dh_1 is exactly w^(T-1): pure geometric growth or decay.
print("Linear recurrence: dh_T/dh_1 = w^(T-1)")
for w in [0.5, 0.9, 1.0, 1.1, 2.0]:
    print(f"  w = {w:>4}: T=5 -> {w**4:.4f}   T=20 -> {w**19:.3e}   T=50 -> {w**49:.3e}")

# Part 2: the real thing, h[t] = tanh(w * h[t-1] + x[t]).
# Each backprop step multiplies by  w * tanh'(pre) = w * (1 - h[t]^2)  <= w.
print("\ntanh recurrence: measured |dh_T/dh_1| (product of per-step factors)")
for w in [0.9, 1.1, 2.0]:
    h = 0.0
    xs = np.random.randn(50) * 0.1
    factors = []
    for t in range(50):
        h = np.tanh(w * h + xs[t])
        factors.append(w * (1.0 - h ** 2))
    prods = np.cumprod(factors)
    print(f"  w = {w:>4}: T=5 -> {abs(prods[4]):.4f}   "
          f"T=20 -> {abs(prods[19]):.3e}   T=50 -> {abs(prods[49]):.3e}")

## Gradient clipping in action


*Expected output starts with:* `--- no clipping (largest raw grad norm seen: inf)`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Exploding gradients in practice, and what clipping does. A vanilla RNN
# reads 60 random numbers and must output their running total -- a task
# that pushes the recurrent weights toward instability, because holding a
# sum means the state must not decay. Same data, same initialization, same
# learning rate; the only difference between the two runs is gradient-norm
# clipping. Predicting 0 always would give a loss near 60 (the variance of
# a sum of 60 standard normals), so 60 is the "learned nothing" baseline.

T, B, H = 60, 16, 32

def run(clip):
    torch.manual_seed(0)
    rnn = nn.RNN(1, H, batch_first=True)
    head = nn.Linear(H, 1)
    params = list(rnn.parameters()) + list(head.parameters())
    opt = torch.optim.SGD(params, lr=0.1)
    max_gnorm, losses = 0.0, []
    for step in range(1, 401):
        x = torch.randn(B, T, 1)
        target = x.sum(dim=1)
        out, _ = rnn(x)
        pred = head(out[:, -1, :])
        loss = ((pred - target) ** 2).mean()
        opt.zero_grad(); loss.backward()
        gnorm = torch.nn.utils.clip_grad_norm_(params, clip).item()
        max_gnorm = max(max_gnorm, gnorm)
        opt.step()
        losses.append(loss.item())
    return losses, max_gnorm

for clip, name in [(float("inf"), "no clipping"), (1.0, "clip at 1.0")]:
    losses, max_gnorm = run(clip)
    print(f"--- {name} (largest raw grad norm seen: {max_gnorm:.1f})")
    t = torch.tensor(losses)
    for lo in range(0, 400, 100):
        print(f"  steps {lo + 1:>3}-{lo + 100:>3}: median loss = {t[lo:lo + 100].median().item():>12.4f}")

## Measuring the difference


*Expected output starts with:* `   T         RNN        LSTM   LSTM fb=1`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# How much gradient reaches the FIRST input token from a loss on the LAST
# hidden state, as the sequence gets longer? Vanilla RNN vs LSTM vs LSTM
# with the forget-gate bias initialized to 1 (the classic trick).

d = 32  # hidden size and input size

def grad_to_first_input(cell_type, T):
    torch.manual_seed(0)
    if cell_type == "rnn":
        cell = nn.RNNCell(d, d)  # tanh nonlinearity
    else:
        cell = nn.LSTMCell(d, d)
        if cell_type == "lstm_fb1":
            with torch.no_grad():
                # gate order in the bias vectors is [input, forget, cell, output]
                cell.bias_ih[d:2 * d].fill_(1.0)
                cell.bias_hh[d:2 * d].fill_(1.0)
    xs = [torch.randn(1, d, requires_grad=True) for _ in range(T)]
    h = torch.zeros(1, d)
    c = torch.zeros(1, d)
    for x in xs:
        if cell_type == "rnn":
            h = cell(x, h)
        else:
            h, c = cell(x, (h, c))
    h.sum().backward()
    return xs[0].grad.norm().item()

print(f"{'T':>4}  {'RNN':>10}  {'LSTM':>10}  {'LSTM fb=1':>10}")
for T in [5, 10, 20, 40, 80]:
    row = [grad_to_first_input(k, T) for k in ["rnn", "lstm", "lstm_fb1"]]
    print(f"{T:>4}  {row[0]:>10.2e}  {row[1]:>10.2e}  {row[2]:>10.2e}")

## Try it yourself

1. In the scalar-recurrence script, set `w = 1.0` and rerun the `tanh` part. The linear analysis says gradients should be preserved — what actually happens, and why?
2. Extend the RNN-vs-LSTM script with a GRU column (`nn.GRUCell`). Where does it fall between the vanilla RNN and the LSTM at `T = 40` and `T = 80`?
3. In the LSTM columns, try forget-gate biases of −1 (gates mostly closed) and +3 (almost fully open). Predict the ordering of gradient norms at `T = 80` before you run.
4. Change the loss from `h.sum()` on the last state to a sum over *all* hidden states. Explain why the measured gradient at the first input changes the way it does.
5. In the truncated-BPTT experiment, set `LAG = 8` and rerun the same windows. Predict which windows can now learn the task before running, then check whether the window-8 run behaves like the old window-4 run and explain why it must.
6. In the clipping experiment, replace the vanilla `nn.RNN` with `nn.LSTM` (keep clipping on) and compare the loss trajectory. How much of the remaining difficulty was explosion, and how much was vanishing?


---

Full discussion of everything above: [S13 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s13/).
